<a href="https://colab.research.google.com/github/LordCastelll/sinintermediarios-contraentrega/blob/main/sinintermediarios_contraentrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd
import glob
import os

# 1. Montar Google Drive (ya lo tenías, lo dejamos por si acaso)
print("Montando Google Drive...")
drive.mount('/content/drive')

# 2. BÚSQUEDA AUTOMÁTICA DE LA CARPETA
# En lugar de adivinar la ruta, buscaremos la carpeta 'Sin_Intermediarios' en todo tu Drive
ruta_carpeta = None
for root, dirs, files in os.walk('/content/drive/MyDrive/'):
    for d in dirs:
        if d == 'Sin_Intermediarios':
            ruta_carpeta = os.path.join(root, d)
            break

if ruta_carpeta:
    print(f"✅ Carpeta encontrada en: {ruta_carpeta}")
else:
    print("❌ No se encontró la carpeta 'Sin_Intermediarios'. Verifica el nombre exacto.")
    # Si no la encuentra, usamos la ruta por defecto para que no falle el resto
    ruta_carpeta = '/content/drive/MyDrive/Sin_Intermediarios/'

# 3. DEFINIR LAS VARIABLES DE BÚSQUEDA
patron_fecha = os.path.join(ruta_carpeta, 'ordenes_*_*_*_contraentregas.csv')
archivos_fecha = glob.glob(patron_fecha)

patron_prueba = os.path.join(ruta_carpeta, 'ordenes_prueba_contraentregas.csv')
archivos_prueba = glob.glob(patron_prueba)

print(f"Archivos con fecha encontrados: {len(archivos_fecha)}")
print(f"Archivos de prueba encontrados: {len(archivos_prueba)}")

# 4. LÓGICA PRINCIPAL
archivo_a_cargar = None

if archivos_fecha:
    print(f"\n📂 Se encontraron {len(archivos_fecha)} archivos con fecha en la carpeta.")

    fecha_input = input("Por favor, ingresa la fecha a procesar (ej: 25-10-23): ")
    fecha_limpia = fecha_input.replace('-', '').replace('_', '').replace('/', '').strip()

    archivos_filtrados = [f for f in archivos_fecha if fecha_limpia in f]

    if archivos_filtrados:
        archivo_a_cargar = archivos_filtrados[0]
        print(f"✅ Archivo encontrado para la fecha indicada: {os.path.basename(archivo_a_cargar)}")
    else:
        print(f"⚠️ No se encontró un archivo con la fecha '{fecha_input}'. Se usará el de pruebas si existe.")

if archivo_a_cargar is None and archivos_prueba:
    archivo_a_cargar = archivos_prueba[0]
    print(f"\n✅ Cargando archivo de PRUEBAS (Fallback): {os.path.basename(archivo_a_cargar)}")

# 5. CARGAR EL ARCHIVO
if archivo_a_cargar:
    df = pd.read_csv(archivo_a_cargar)
    print(f"\n📊 Filas iniciales cargadas: {len(df)}")
    print("\nPrimeras 5 filas:")
    display(df.head())
else:
    print("\n⚠️ No se pudo cargar ningún archivo. Revisa los nombres en tu Drive.")


Montando Google Drive...
Mounted at /content/drive
✅ Carpeta encontrada en: /content/drive/MyDrive/Sin_Intermediarios
Archivos con fecha encontrados: 0
Archivos de prueba encontrados: 1

✅ Cargando archivo de PRUEBAS (Fallback): ordenes_prueba_contraentregas.csv

📊 Filas iniciales cargadas: 18

Primeras 5 filas:


,pedido_id,cliente,es_miembro,ciudad_destino,metodo_envio,producto_reportado,cantidad,valor_cobrado_actual
0,PED-1001,Camila Ríos,Si,Bogotá,Estándar,Whey choco 2lb,2,311000
1,PED-1002,Julián Torres,No,Medellín,Estándar,Creatina monohidratada 300g,1,96000
2,PED-1003,Valentina Gómez,Si,Cali,Estándar,Multivitaminico mujer,1,68000
3,PED-1004,Andrés Peña,Si,Bucaramanga,Estándar,Colageno limon 300g,1,187000
4,PED-1004,Andrés Peña,Si,Bucaramanga,Estándar,Omega 3 x120,1,187000


In [ ]:
import difflib # Importamos la librería para el matching inteligente

# --- 1. CATÁLOGO MAESTRO COMPLETO (14 Productos) ---
catalogo_maestro = {
    "Whey Protein Concentrado Chocolate 908 g": 145000,
    "Whey Protein Concentrado Vainilla 908 g": 145000,
    "Whey Protein Isolate Fresa 1.13 kg": 200000,
    "Whey Protein Isolate Fresa 2.27 kg": 310000,
    "Creatina Monohidratada Neutro 300 g": 75000,
    "Pre-Entreno Explosive Sandía 300 g": 110000,
    "BCAA 2:1:1 Mango 400 g": 85000,
    "Glutamina Neutro 300 g": 80000,
    "Multivitamínico Hombre x60 cápsulas": 58000,
    "Multivitamínico Mujer x60 cápsulas": 58000,
    "Colágeno Hidrolizado Limón 300 g": 92000,
    "Omega 3 1000 mg x120 cápsulas": 70000,
    "Quemador Termogénico Black x90 cápsulas": 98000,
    "Barra Proteica Caja x12 Choco Chip": 65000
}

# --- 2. MOTOR DE ASOCIACIÓN INTELIGENTE (Fuzzy Matching) ---
# Esta función busca el producto del catálogo que más se parezca al nombre del CSV
def buscar_producto_similar(nombre_csv):
    # Si hay una coincidencia exacta, la usa
    if nombre_csv in catalogo_maestro:
        return nombre_csv, catalogo_maestro[nombre_csv], 100.0

    # Buscamos las mejores coincidencias (umbral mínimo de 30% de similitud)
    coincidencias = difflib.get_close_matches(nombre_csv, catalogo_maestro.keys(), n=1, cutoff=0.3)

    if coincidencias:
        mejor_match = coincidencias[0]
        similitud = difflib.SequenceMatcher(None, nombre_csv, mejor_match).ratio() * 100
        return mejor_match, catalogo_maestro[mejor_match], round(similitud, 2)

    return "⚠️ NO ENCONTRADO", 0, 0.0

# --- 3. LIMPIEZA PROFUNDA Y CÁLCULO DE REGLAS DE NEGOCIO ---

df_limpio = df.copy()

# A. Limpieza básica de textos
for col in df_limpio.select_dtypes(include=['object']).columns:
    df_limpio[col] = df_limpio[col].astype(str).str.strip()

df_limpio['es_miembro'] = df_limpio['es_miembro'].str.upper().replace({'S': 'SI', 'N': 'NO'})

# B. Estandarizar Ciudades
mapeo_ciudades = {
    'BOGOTA': 'Bogotá', 'BOGOTÁ': 'Bogotá',
    'CALI': 'Cali',
    'MEDELLIN': 'Medellín', 'MEDELLÍN': 'Medellín'
}
df_limpio['ciudad_destino'] = df_limpio['ciudad_destino'].str.upper().map(mapeo_ciudades).fillna(df_limpio['ciudad_destino'].str.title())

# C. Estandarizar Métodos de Envío
mapeo_envios = {
    'ESTANDAR': 'Estandar', 'ESTÁNDAR': 'Estandar',
    'MISMO DIA': 'Mismo Dia', 'MISMO DÍA': 'Mismo Dia',
    'DIA SIGUIENTE': 'Dia Siguiente', 'DÍA SIGUIENTE': 'Dia Siguiente'
}
df_limpio['metodo_envio'] = df_limpio['metodo_envio'].str.upper().map(mapeo_envios).fillna('Revisar')

# D. APLICAR MATCHING A LOS PRODUCTOS
# Creamos una columna temporal con los resultados
resultados_match = df_limpio['producto_reportado'].apply(buscar_producto_similar)

df_limpio['nombre_catalogo'] = [r[0] for r in resultados_match]
df_limpio['precio_unitario'] = [r[1] for r in resultados_match]
df_limpio['similitud_%'] = [r[2] for r in resultados_match]

# Alerta si la similitud es baja (menos de 60%)
df_limpio['estado_match'] = df_limpio['similitud_%'].apply(
    lambda x: '✅ Automático' if x >= 60 else '⚠️ Revisión Manual'
)

# E. Cálculo de subtotal por línea
df_limpio['subtotal_linea'] = df_limpio['precio_unitario'] * df_limpio['cantidad']

# F. Agrupación por pedido (Para los pedidos con varios productos)
df_pedidos = df_limpio.groupby('pedido_id').agg({
    'cliente': 'first',
    'es_miembro': 'first',
    'ciudad_destino': 'first',
    'metodo_envio': 'first',
    'subtotal_linea': 'sum',
    'valor_cobrado_actual': 'first'
}).reset_index()

df_pedidos.rename(columns={'subtotal_linea': 'subtotal_calculado'}, inplace=True)

# G. Aplicación de Reglas de Negocio (Envío y Recargo)
def aplicar_reglas(row):
    subtotal = row['subtotal_calculado']
    es_miembro = row['es_miembro'] == 'SI'
    ciudad = row['ciudad_destino']
    metodo = row['metodo_envio']

    # Regla de Envío Gratis (estrictamente mayor a 200.000)
    if es_miembro and subtotal > 200000:
        costo_envio = 0
    else:
        if ciudad in ['Bogotá', 'Cali', 'Medellín']:
            if metodo == 'Estandar': costo_envio = 11000
            elif metodo == 'Dia Siguiente': costo_envio = 15000
            elif metodo == 'Mismo Dia': costo_envio = 16000
            else: costo_envio = 0
        else:
            costo_envio = 15000 # Resto del país

    recargo_fijo = 10000
    total = subtotal + costo_envio + recargo_fijo
    diferencia = row['valor_cobrado_actual'] - total

    return pd.Series([costo_envio, recargo_fijo, total, diferencia])

df_pedidos[['costo_envio', 'recargo_fijo', 'total_correcto', 'diferencia']] = df_pedidos.apply(aplicar_reglas, axis=1)

# --- 4. DIAGNÓSTICO Y RESULTADOS ---
print("--- 🕵️‍♂️ AUDITORÍA DE PRODUCTOS (MATCHING) ---")
columnas_match = ['producto_reportado', 'nombre_catalogo', 'precio_unitario', 'similitud_%', 'estado_match']
display(df_limpio[columnas_match].drop_duplicates().sort_values(by='similitud_%', ascending=False))

print("\n--- 📊 PEDIDOS EVALUADOS (Fórmula visible) ---")
columnas_reporte = ['pedido_id', 'cliente', 'es_miembro', 'ciudad_destino', 'metodo_envio',
                    'subtotal_calculado', 'costo_envio', 'recargo_fijo', 'total_correcto',
                    'valor_cobrado_actual', 'diferencia']

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(df_pedidos[columnas_reporte])

print(f"\n✅ Total de pedidos evaluados: {len(df_pedidos)}")
print(f"⚠️ Pedidos que requieren corrección: {len(df_pedidos[df_pedidos['diferencia'] != 0])}")

# --- 5. CAPA DE DECISIÓN HUMANA (Para Colab) ---
print("\n--- 🛑 REVISIÓN MANUAL REQUERIDA ---")
productos_dudosos = df_limpio[df_limpio['estado_match'] == '⚠️ Revisión Manual']['producto_reportado'].unique()

for prod in productos_dudosos:
    sugerencia = df_limpio[df_limpio['producto_reportado'] == prod]['nombre_catalogo'].iloc[0]
    precio = df_limpio[df_limpio['producto_reportado'] == prod]['precio_unitario'].iloc[0]

    print(f"\nEl producto '{prod}' se asoció a '{sugerencia}' (${precio}).")
    respuesta = input("¿Es correcto? (S/N): ")

    if respuesta.upper() != 'S':
        print("❌ Ok, tendrás que agregarlo manualmente al diccionario.")
    else:
        print("✅ Confirmado.")


--- 🕵️‍♂️ AUDITORÍA DE PRODUCTOS (MATCHING) ---


,producto_reportado,nombre_catalogo,precio_unitario,similitud_%,estado_match
1,Creatina monohidratada 300g,Creatina Monohidratada Neutro 300 g,75000,83.87,✅ Automático
16,Glutamina 300g,Glutamina Neutro 300 g,80000,77.78,✅ Automático
5,Pre entreno explosive sandia,Pre-Entreno Explosive Sandía 300 g,110000,74.19,✅ Automático
9,Barra proteica caja x12,Barra Proteica Caja x12 Choco Chip,65000,73.68,✅ Automático
10,Multivitaminico hombre,Multivitamínico Hombre x60 cápsulas,58000,70.18,✅ Automático
2,Multivitaminico mujer,Multivitamínico Mujer x60 cápsulas,58000,69.09,✅ Automático
3,Colageno limon 300g,Colágeno Hidrolizado Limón 300 g,92000,66.67,✅ Automático
12,whey isolate fresa,Whey Protein Isolate Fresa 2.27 kg,310000,61.54,✅ Automático
6,whey isolate fresa 5 libras,Whey Protein Isolate Fresa 2.27 kg,310000,59.02,⚠️ Revisión Manual
4,Omega 3 x120,Omega 3 1000 mg x120 cápsulas,70000,58.54,⚠️ Revisión Manual



--- 📊 PEDIDOS EVALUADOS (Fórmula visible) ---


,pedido_id,cliente,es_miembro,ciudad_destino,metodo_envio,subtotal_calculado,costo_envio,recargo_fijo,total_correcto,valor_cobrado_actual,diferencia
0,PED-1001,Camila Ríos,SI,Bogotá,Estandar,290000,0,10000,300000,311000,11000
1,PED-1002,Julián Torres,NO,Medellín,Estandar,75000,11000,10000,96000,96000,0
2,PED-1003,Valentina Gómez,SI,Cali,Estandar,58000,11000,10000,79000,68000,-11000
3,PED-1004,Andrés Peña,SI,Bucaramanga,Estandar,162000,15000,10000,187000,187000,0
4,PED-1005,Mariana Duarte,NO,Bogotá,Mismo Dia,220000,16000,10000,246000,241000,-5000
5,PED-1006,Felipe Salazar,SI,Medellín,Estandar,310000,0,10000,320000,320000,0
6,PED-1007,Laura Jiménez,SI,Cali,Dia Siguiente,250000,0,10000,260000,275000,15000
7,PED-1008,Diego Herrera,NO,Barranquilla,Estandar,195000,15000,10000,220000,220000,0
8,PED-1009,Camilo Restrepo,SI,Bogotá,Estandar,133000,11000,10000,154000,144000,-10000
9,PED-1010,Natalia Cárdenas,SI,Medellín,Estandar,310000,0,10000,320000,200000,-120000



✅ Total de pedidos evaluados: 14
⚠️ Pedidos que requieren corrección: 9

--- 🛑 REVISIÓN MANUAL REQUERIDA ---

El producto 'Whey choco 2lb' se asoció a 'Whey Protein Concentrado Chocolate 908 g' ($145000).
¿Es correcto? (S/N): S
✅ Confirmado.

El producto 'Omega 3 x120' se asoció a 'Omega 3 1000 mg x120 cápsulas' ($70000).
¿Es correcto? (S/N): S
✅ Confirmado.

El producto 'whey isolate fresa 5 libras' se asoció a 'Whey Protein Isolate Fresa 2.27 kg' ($310000).
¿Es correcto? (S/N): S
✅ Confirmado.

El producto 'bcaa mango' se asoció a 'BCAA 2:1:1 Mango 400 g' ($85000).
¿Es correcto? (S/N): S
✅ Confirmado.

El producto 'glutamina' se asoció a 'Glutamina Neutro 300 g' ($80000).
¿Es correcto? (S/N): S
✅ Confirmado.

El producto 'creatina' se asoció a 'Creatina Monohidratada Neutro 300 g' ($75000).
¿Es correcto? (S/N): S
✅ Confirmado.

El producto 'Vitamina C efervescente' se asoció a 'Glutamina Neutro 300 g' ($80000).
¿Es correcto? (S/N): N
❌ Ok, tendrás que agregarlo manualmente al diccio